In [1]:
import torch 
import torch.nn as nn 
import torch.optim as optim 

import torchvision
from torchvision.datasets import CIFAR10

In [2]:
from torch.utils.data import DataLoader
import torchvision.transforms as transforms

transform = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize((0.5 , 0.5 , 0.5) , (0.5 , 0.5 , 0.5)),
])

trainset =CIFAR10(root = "./Data" , train = True , download = True , transform = transform) 
testset =CIFAR10(root = "./Data" , train = False , download = True , transform = transform) 

100%|██████████| 170M/170M [11:35<00:00, 245kB/s]  


In [3]:
trainloder = DataLoader(
    trainset , 
    batch_size = 64 , 
    shuffle = True 
)

testloder = DataLoader(
    testset , 
    batch_size = 64
)

In [4]:
class CNN(nn.Module) :
    def __init__ (self) :
        super(CNN , self).__init__()

        self.conv_layers = nn.Sequential(
            nn.Conv2d(3 , 32 , kernel_size = 3 , padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2 , 2),

            nn.Conv2d(32 , 64 , kernel_size = 3 , padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2 , 2),

            nn.Conv2d(64 , 128 , kernel_size = 3 , padding = 1),
            nn.ReLU(),
            nn.MaxPool2d(2 , 2)
        )

        self.fc_layers = nn.Sequential(
            nn.Linear( 4*4*128 , 256),
            nn.ReLU(),
            nn.Linear( 256 , 10 )
        )

    def forward(self , x) :
        x = self.conv_layers(x)
        x = x.view(x.size(0) , -1)
        x = self.fc_layers(x)

        return x
        

In [5]:
model = CNN()

In [6]:
critrion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters())

In [7]:
epochs = 10

for epoch in range(epochs):
    epoch_tranning_loss = 0.0

    for images , labels in trainloder:
        optimizer.zero_grad()
        
        output = model.forward(images)
        loss = critrion(output , labels)

        loss.backward()
        optimizer.step()

        epoch_tranning_loss += loss.item()

    print(f"Epoch = {epoch+1}/{epochs} and Loss = {epoch_tranning_loss/len(trainloder)}")

Epoch = 1/10 and Loss = 1.3800835822091992
Epoch = 2/10 and Loss = 0.9513831027328511
Epoch = 3/10 and Loss = 0.7771185280569374
Epoch = 4/10 and Loss = 0.6531887182494258
Epoch = 5/10 and Loss = 0.5477425997214549
Epoch = 6/10 and Loss = 0.46646370910241475
Epoch = 7/10 and Loss = 0.38211522811590254
Epoch = 8/10 and Loss = 0.310005398824468
Epoch = 9/10 and Loss = 0.2515480677356653
Epoch = 10/10 and Loss = 0.1989661082768303


In [10]:
correct_labels = 0
total_labels = 0

model.eval()

with torch.no_grad():
    for images , labels in testloder:
        outputs = model.forward(images)
        _,predicted = torch.max(outputs , 1)

        correct_labels += (predicted == labels).sum().item()
        total_labels += labels.size(0)

print(f"Accuracy :-> {(correct_labels / total_labels) * 100}")

Accuracy :-> 74.98
